In [1]:
import pandas as pd

# Grab data from local
df = pd.read_csv("/Volumes/trevor_external/capstone-backend/environmental_impact_of_food.csv")


df

,entity,ghg_kg_co2eq_per_kg,land_use_m2_per_kg,freshwater_withdrawals_l_per_kg,eutrophying_emissions_g_po4eq_per_kg,scarcity_weighted_water_l_per_kg
0,Apples,0.43,0.63,180.1,1.45,12948.6
1,Bananas,0.86,1.93,114.5,3.29,661.9
2,Barley,1.18,1.11,17.1,2.33,696.4
3,Beef (beef herd),99.48,326.21,1451.2,301.41,34732.5
4,Beef (dairy herd),33.30,43.24,2714.3,365.29,119805.2
5,Beet Sugar,1.81,1.83,217.7,5.41,9493.3
6,Berries & Grapes,1.53,2.41,419.6,6.12,21162.1
7,Brassicas,0.51,0.55,119.4,5.01,8455.1
8,Cane Sugar,3.20,2.04,620.1,16.92,16438.6
9,Cassava,1.32,1.81,0.0,0.69,0.0


### Composite Score Methodology
1) First, the data is too wide. The distance between the lowest C02 values (apples) and the highest (Beef (Beef Herd)) are so high, that it we perform mid max normalization, the smaller grouped values are nearly indistinguishable from one another. Eggs look to almost be the same as apples in terms of C02 output, even though eggs have a ghg_kg_co2eq_per_kg slightly over 10x higher than apples. Subsequently, we need to shrink the width of the data and in this case I decided to go with logarithmic transformation to do just this. 
<br></br>
2) Next we need to make it so that we can combine all of these totally varying units into a single finalized
 score value. It would be illogical to simply add ghg_kg_co2eq_per_kg to land_use_m2_per_kg. Subsequently, min max normalization becomes our knight in shining armor here, allowing us to put every value, despite their unit differences, between a scale ranging from 0-1. 
 <br></br>
3) Now, we must deal with the issue of these 5 different environmental impact measures having differing levels of severity in their respective contribution to anthropogenic environmental impact. Subsequently, we need to weigh the respective variables and to do so, I ended up going with weights from the lovely research team that put [this](https://pmc.ncbi.nlm.nih.gov/articles/PMC9700787/ "Go to NIH publication") research paper together. They went ahead and already created the weighing factors based off of the data from the EU's Product Environmental Footprint (PEF) methodology. I did run into two issues doing this: 1) We have two water-related columns while the PEF only has one. I decided to drop freshwater_withdrawals_l_per_kg and to leave in scarcity_weighted_water_l_per_kg since the former is just the raw volume while the latter is already adjusted for how stressed a water source is 2) the farmed fish and farmed prawn values should way more heavily on marine/freshwater eutrophication, which the PEF structure agrees with, however our OWID dataset only contains a single combined number here. Subsequently, as a shortcut for this initial version, I simply added the three different eutrophication category weights together to create a single weight. 
<br></br>
4) Lastly, our means of combining all the values is with geometric mean. I chose geometric mean as opposed to arithmetic mean because in our case, we do not want a single poor performing metric to be able to hide behind a 4 other well-ranged metrics. With arithmetic mean, the value can only be brought down or up by the amount of the outlier itself. This ends up capping the negative effect of a single outlier considerably due to the limitations of addition here. However, geometric mean uses multiplication prior to taking the n square root of the product. Subsequently, a single significantly poor performing outlier for the otherwise group of good values can significantly lower and or raise the resulting value. Essentially, if a food does well in every category except say ghg_kg_co2eq_per_kg, we want the final score to accurately reflect that rather than portraying the food as environmentally less harmful overall. 

### Logarithmic Transformation

In [2]:
import numpy as np

# Transform the numerical columns into float types so they can be ran through numpy
df.iloc[:, 1:] = df.iloc[:, 1:].astype(float)

### Log Transform
# First we need to compress the values because they are too spread out which will scatter our values too much
# in min maxing transformation. For this we will first perform log transformation
transform_df = df.copy()

# Drop the freshwater_withdrawals_l_per_kg column prior to beginning our transformations 
transform_df.drop(columns=["freshwater_withdrawals_l_per_kg"], inplace=True)

# Isolate the float-type columns we want to log transform
columns_to_transform = transform_df.columns[1:]

# Handle genuine zeros before logging: replace with half the smallest
# nonzero value in that column (standard technique — avoids log(0) while
# still correctly treating the food as "best in dataset" on that metric)
raw_for_log = df[columns_to_transform].copy()
for col in columns_to_transform:
    nonzero_min = raw_for_log.loc[raw_for_log[col] > 0, col].min()
    raw_for_log[col] = raw_for_log[col].replace(0, nonzero_min / 2)


# Perform the transformation
transform_df.loc[:, columns_to_transform] = np.log(raw_for_log)

### Min Max Normalization

In [7]:
from sklearn.preprocessing import MinMaxScaler

# Initialize our scaler
scaler = MinMaxScaler(feature_range=(0.01, 1))



# Use the same designated columns from previous section
transform_df[columns_to_transform] = scaler.fit_transform(transform_df[columns_to_transform])



### Apply the Weights and Calculate Geometric Mean


In [ ]:


weights = { 
        "ghg_kg_co2eq_per_kg": 0.4472,
        "land_use_m2_per_kg" : 0.1697,
        "eutrophying_emissions_g_po4eq_per_kg": 0.2011,
        "scarcity_weighted_water_l_per_kg": 0.1820
        }


def weighted_geometric_mean(row): 
        log_sum = sum(w * np.log(row[col]) for col, w in weights.items())
        return np.exp(log_sum)

# Create the new feature and add it to the original df
df["environmental_impact_score"] = transform_df.iloc[:, 1:].apply(weighted_geometric_mean, axis=1) * 100

df.head()

               entity  ghg_kg_co2eq_per_kg  land_use_m2_per_kg  \
0              Apples                 0.43                0.63   
1             Bananas                 0.86                1.93   
2              Barley                 1.18                1.11   
3    Beef (beef herd)                99.48              326.21   
4   Beef (dairy herd)                33.30               43.24   
5          Beet Sugar                 1.81                1.83   
6    Berries & Grapes                 1.53                2.41   
7           Brassicas                 0.51                0.55   
8          Cane Sugar                 3.20                2.04   
9             Cassava                 1.32                1.81   
10             Cheese                23.88               87.79   
11       Citrus Fruit                 0.39                0.86   
12             Coffee                28.53               21.62   
13     Dark Chocolate                46.65               68.96   
14        

### RAG Set Up

In [ ]:
import anthropic
import re

# Initialize our client
client = anthropic.Anthropic()


FOOD_NAMES = df["entity"].tolist()

In [ ]:
# Create our methodology note so that the model can accurately respond to narrative-framed questions
# EX: "How can I trust this data?"

METHODOLOGY_NOTE = """
This data is drawn from Poore & Nemecek (2018), a meta-analysis of ~38,000 farms
across 119 countries, published by Our World in Data. Figures are GLOBAL AVERAGES
per kilogram of product, pinned to a 2010 reference year — not a specific grocery
item, country, or production method (e.g. grass-fed vs. feedlot beef). The
Environmental Impact Score combines four metrics (GHG emissions, land use,
eutrophying emissions, and scarcity-weighted water use) using a weighted
geometric mean, with weights sourced from the EU's Product Environmental
Footprint (PEF) methodology. Treat results as directional and comparative,
not precise or personalized.
"""

In [ ]:
import re
### Food Keyword Matching
# Find if any foods from our list of food entities in our data can be found within the user's query

# Extract the matching foods from our food list from the user's query
def match_foods(query): 
    # Extract all full words from user query, returns list of words
    q_words = re.findall(r"[a-z+]", query.lower())
    
    matched = []
    # Iterate through all food category names within our df
    for name in FOOD_NAMES: 
        # Drop any abbreviated specifications for food like "Beef (dairy)" -> "beef"
        base = re.sub(r"/(.*?/)", "", name)
        
        # Set iterator that returns a all food names without any excess characters like trailing spaces
        name_words = {w for w in re.findall(r"[a-z]+", base) if len(w) > 2}
        
        # A conditional that checks if the food group name is in the user's query
        # or if there are other values-in-common in or food name set and the list of words from the query
        if name.lower() in query.lower() or name_words & q_words: 
            matched.append(name)
        

In [ ]:
### Router
# This decides what recourses Claude needs in order to effectively respond to a user. It does this with plain string
# search to save a model call. 
# Structured = Requires quantitative data to respond. Look up our dataset
# Narrative = Requires our methodology note to effectively respond
# Both = Both the data and our methodology note to respond

STRUCTURED_KEYWORDS = ["compare", " vs ", "versus", "higher", "lower", "highest",
                        "lowest", "rank", "worse", "better", "which is", "more than"]
NARRATIVE_KEYWORDS = ["why", "should i", "how reliable", "trust", "accurate",
                    "methodology", "confidence", "source"]

def route_query(query): 
    # Standardize the casing of the user query
    q = query.lower()
    # Return true if any of the structured keywords are in the user query or match_food returned list contains any values
    wants_structured = any(k in q for k in STRUCTURED_KEYWORDS) or len(match_foods(query)) >= 1
    # Return true if any narrative keywords found within the user query
    wants_narrative = any(k in q for k in NARRATIVE_KEYWORDS)
    if wants_structured and wants_narrative: 
        return "both"
    return "narrative" if wants_narrative else "structured"

In [ ]:
# Context Assembly
# Takes the label from route_query and translates it to text so that Claude can read it. Claude can only 
# read text, so the query and any required information from our data must be first converted to text
# prior to being sent to Claude

DISPLAY_COLS = ["entity", "ghg_kg_co2eq_per_kg", "land_use_m2_per_kg",
                "eutrophying_emissions_g_po4eq_per_kg",
                "scarcity_weighted_water_l_per_kg", "environmental_impact_score"]

# If applicable, get the relevant rows from our data and then convert them to a string to appended to our 
# context in the next function
def structured_context(query): 
    # Get our list of matched foods from the user query
    foods = match_foods(query)
    # If no foods found, return this string context to say so
    if not foods: 
        return "No specific foods matched in the dataset for this query."
    # Create a subset of rows that contain the data corresponding to the matched foods from the user query
    subset = df[df["entity"]].isin(foods)[DISPLAY_COLS]
    # Convert the data to a string 
    return "RELEVANT DATA:\n" + subset.to_string(index=False)

# Build the final context to provide our model
def build_context(query, route): 
    parts = []
    # If we need structured data to accurately respond, create it then append it
    if route in ("structured", "both"): 
        parts.append(structured_context(query))
    # If we need methodology notes, add them
    if route in ("narrative", "both"): 
        parts.append("METHODOLOGY NOTES: \n" + METHODOLOGY_NOTE)
    # Join all parts with double line spacing between all bodies
    return "\n\n".join(parts)
    


In [ ]:
# Create the prompt for the model and send it

SYSTEM_PROMPT = (
    "You answer questions about the environmental impact of food using the "
    "Poore & Nemecek (2018) / Our World in Data dataset and the Environmental "
    "Impact Score computed from it. Use the provided data when given, and be "
    "precise with numbers in comparisons. When a question is interpretive or "
    "about reliability, weave in the methodology caveats naturally rather than "
    "overclaiming precision. Keep answers concise and conversational."
)


# Put it all together
def answer_question(query): 
    route = route_query(query)
    context = build_context(query, route)
    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024, 
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"{context}\n\nQuestion: {query}"}]
        
    )
    return message.content[0].text